# Example: Replaying a Single Index Model with EWLS
In this example, we compare a frozen Single Index Model estimate with exponentially weighted least squares when an asset's market exposure changes.

> __Learning Objectives:__
>
> By the end of this example, you will be able to:
>
> * __Implement the EWLS recursion:__ Update sufficient statistics and recover online SIM coefficients.
> * __Interpret estimator memory:__ Convert a half-life into a forgetting factor and compare responsiveness with stability.
> * __Evaluate parameter tracking:__ Measure estimation error before and after a structural change.

Let's test how quickly an online estimator recognizes that yesterday's calibration is stale.
___


## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading the packages used in this example.

> __Include:__ The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates `Include.jl` in the notebook's global scope. The file activates the course environment, defines notebook-relative paths, and loads the required packages.

Let's set up the code environment:

The reusable portfolio algorithms in this example are provided by the local [`VLQuantitativeFinancePackage.jl`](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/) package.


In [ ]:
include(joinpath(@__DIR__, "Include.jl"));


For additional information, see the [Julia documentation](https://docs.julialang.org/en/v1/) and the [CHEME 5660 documentation](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/).

___


## Task 1: Generate a Changing-Beta SIM
The market return $g_{m,t}$ drives the asset return
$$
g_{i,t}=\alpha+\beta_t g_{m,t}+\varepsilon_t.
$$
The true beta changes from 0.7 to 1.5 halfway through the sample.


In [ ]:
Random.seed!(5660);
T = 600;
change_day = 300;
market = rand(Normal(0.07/252, 0.18/sqrt(252)), T);
true_beta = vcat(fill(0.7, change_day), fill(1.5, T-change_day));
alpha_true = 0.015/252;
noise = rand(Normal(0.0, 0.12/sqrt(252)), T);
asset = alpha_true .+ true_beta .* market .+ noise;

calibration = 1:200;
X = hcat(ones(length(calibration)), market[calibration]);
frozen = X\asset[calibration];
frozen_alpha, frozen_beta = frozen;


## Task 2: Implement EWLS
With forgetting factor $\lambda=2^{-1/h}$ for half-life $h$, maintain
$$
\mathbf A_t=\lambda\mathbf A_{t-1}+\mathbf x_t\mathbf x_t^{\mathsf T},
\qquad
\mathbf b_t=\lambda\mathbf b_{t-1}+\mathbf x_ty_t,
$$
then solve $\widehat{\boldsymbol\theta}_t=\mathbf A_t^{-1}\mathbf b_t$. A ridge seed keeps the early system nonsingular.


In [ ]:
half_lives = [20.0, 60.0, 120.0];
paths = Dict(h => ewls_path(market, asset; half_life=h) for h in half_lives);


## Task 3: Compare Tracking Error
We compare beta mean-squared error during the stable pre-change interval and during the first 100 observations after the change.


In [ ]:
tracking = DataFrame(method=String[], prechange_mse=Float64[],
    postchange_mse=Float64[]);
push!(tracking, ("Frozen OLS",
    mean((frozen_beta .- true_beta[201:change_day]).^2),
    mean((frozen_beta .- true_beta[(change_day+1):(change_day+100)]).^2)));
for h in half_lives
    beta_path = paths[h][:,2];
    push!(tracking, ("EWLS h=$(Int(h))",
        mean((beta_path[201:change_day] .- true_beta[201:change_day]).^2),
        mean((beta_path[(change_day+1):(change_day+100)] .-
            true_beta[(change_day+1):(change_day+100)]).^2)));
end
pretty_table(tracking; table_format=TextTableFormat(borders=text_table_borders__simple))


In [ ]:
plot(1:T, true_beta, lw=3, c=:black, label="True beta",
    xlabel="Trading day", ylabel="Beta estimate")
hline!([frozen_beta], lw=2, c=:gray, ls=:dash, label="Frozen OLS")
for (index,h) in enumerate(half_lives)
    plot!(1:T, paths[h][:,2], lw=1.8, label="EWLS h=$(Int(h))")
end
vline!([change_day], c=:red, ls=:dot, label="Structural change")


## Summary
This example demonstrated the stability–responsiveness tradeoff in online SIM estimation.

> __Key Takeaways:__
>
> * __Frozen estimates do not diagnose their own staleness:__ They continue reporting the calibration-period beta after the process changes.
> * __Half-life controls memory:__ Shorter half-lives adapt faster but produce noisier estimates.
> * __Online estimation still requires validation:__ Responsiveness alone does not establish that an adaptive trading system improves risk-adjusted performance.

The next example turns these diagnostics into explicit deployment gates.
___

## Disclaimer and Risks
This material is for educational purposes only. Structural changes in real markets are not directly observable and may be confounded with noise, missing variables, and data-quality problems.
